# Обучение учителя ResNet-50 на CIFAR-100 (Colab, T4)

Шаг 2 проекта `cifar100-distillation`: ResNet-50 с нуля (стем conv3x3, без maxpool),
200 эпох, SGD (nesterov, lr 0.1, momentum 0.9, wd 5e-4), батч 128, warmup 1 эпоха + косинус,
AMP (fp16) и `channels_last`. Цель — ≈78–80% top-1 на test.

Запуск ручной: `train.py` сам создаёт задачу ClearML (`Task.init`). Чекпоинт пишется на
Google Drive в `<output_dir>/<task_id>/run_0/checkpoint.pt`, поэтому после обрыва сессии обучение
продолжается повторным запуском с `--continue_task_id <task_id>`.

Перед запуском: **Среда выполнения → Сменить среду выполнения → T4 GPU**, а в **Secrets**
(значок ключа слева) добавить `CLEARML_API_ACCESS_KEY` и `CLEARML_API_SECRET_KEY`
(app.clear.ml → Settings → Workspace → Create new credentials) и открыть к ним доступ ноутбуку.

## 1. Google Drive и GPU

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Нет GPU: смени среду выполнения на T4"

## 2. Код и зависимости

torch/torchvision в Colab уже стоят со сборкой под CUDA — их не переустанавливаем,
ставим только остальное из `requirements.txt`. Обучение идёт отдельным процессом
(`!python train.py`), поэтому предупреждения pip о версиях пакетов самого Colab не мешают.

In [ ]:
import os

REPO_URL = "https://github.com/esgon191/cifar100-distillation"
REPO_DIR = "/content/cifar100-distillation"
BRANCH = "master"

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
!git log -1 --oneline

In [ ]:
!grep -viE '^(torch|torchvision)==' requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt

## 3. ClearML

Ключи берутся из Colab Secrets и передаются через переменные окружения — их наследует
процесс `train.py` (`Task.set_credentials` подействовал бы только на ядро ноутбука).

In [ ]:
from google.colab import userdata

for name in ("CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY"):
    os.environ[name] = userdata.get(name)
os.environ["CLEARML_API_HOST"] = "https://api.clear.ml"
os.environ["CLEARML_WEB_HOST"] = "https://app.clear.ml"
os.environ["CLEARML_FILES_HOST"] = "https://files.clear.ml"

# Проверка авторизации без создания задачи
!python -c "from clearml.backend_api.session import Session; Session(); print('ClearML OK, files server:', Session.get_files_server_host())"

## 4. Параметры запуска

`DRIVE_OUT` — каталог на Drive: в нём для каждой задачи появится папка `<task_id>/run_0`
с `checkpoint.pt` (≈190 МБ: веса + momentum), `summary.json` и `model_final.pt`.
Датасет берётся из ClearML Dataset `cifar100` (files server, ~10 с; кэш в `~/.clearml/cache`),
а не с сервера CIFAR в Торонто (~90 КБ/с).
Nesterov, косинусный график и AMP включены по умолчанию.

In [ ]:
DRIVE_OUT = "/content/drive/MyDrive/cifar100-distillation/out"
SEED = 0

TRAIN_ARGS = (
    f"--arch resnet50 --seed {SEED} --task_name teacher-resnet50-seed{SEED} --tags teacher "
    "--num_train_epochs 200 --learning_rate 0.1 --momentum 0.9 --nesterov "
    "--per_device_train_batch_size 128 --weight_decay 5e-4 --warmup_epochs 1 "
    "--save_every 2 --eval_every 1 "
    f"--dataset_name cifar100 --output_dir {DRIVE_OUT}"
)
os.makedirs(DRIVE_OUT, exist_ok=True)
print(TRAIN_ARGS)

## 5. Первый запуск

В начале лога будет строка `ClearML Task: created new task id=...` — запиши id
(он же имя папки в `DRIVE_OUT`). Время эпохи — в логе и на графике `epoch_time` в ClearML;
по первым эпохам видно, сколько займёт обучение целиком.

In [ ]:
!python -u train.py {TRAIN_ARGS}

## 6. Продолжение после обрыва сессии

Новая сессия: выполнить ячейки 1–4, затем эту, указав id задачи. Аргументы должны
совпадать с первым запуском (`TRAIN_ARGS`). Скрипт продолжит ту же задачу ClearML
(графики дописываются с той же эпохи) и подхватит `DRIVE_OUT/<task_id>/run_0/checkpoint.pt`;
если id неверный, скрипт остановится с ошибкой, а не начнёт обучение заново.
Последние ≤2 эпохи до обрыва будут пересчитаны (`--save_every 2`).

In [ ]:
!ls -lt {DRIVE_OUT}

In [ ]:
TASK_ID = ""  # id задачи ClearML = имя папки в DRIVE_OUT
assert TASK_ID, "Укажи TASK_ID"
!python -u train.py {TRAIN_ARGS} --continue_task_id {TASK_ID}

## 7. После обучения

Итоговые метрики (финальная модель, конец косинуса) — single values `test/top1`,
`test/top5`, `val/*` в задаче ClearML; веса `model_final.pt` загружены на files server
ClearML и лежат на Drive. Перед закрытием ноутбука — дождаться синхронизации Drive.

In [ ]:
drive.flush_and_unmount()

## Приложение: clearml-agent в Colab

Воркер очереди `colab-t4` вынесен в отдельный ноутбук `colab/agent.ipynb` (он же исполняет задачи сетки ёмкости ученика).

In [ ]:
# см. colab/agent.ipynb